# Vision Agents with smolagents


This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Install dependencies and configure the OpenAI API

Install `smolagents` with its OpenAI integration plus the helper packages used by this notebook.

In [1]:
%pip install -q --no-cache-dir 'smolagents[openai]' python-dotenv pillow requests


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Set `OPENAI_API_KEY` in your environment or a `.env` file. If it is missing, the next cell prompts securely. Optionally set `OPENAI_VISION_MODEL`; it defaults to `gpt-4.1-mini`, which supports image input.

In [2]:
import os
from getpass import getpass

from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
if env_file:
    load_dotenv(env_file)

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is required to run this notebook.")

OPENAI_VISION_MODEL = os.getenv("OPENAI_VISION_MODEL", "gpt-4.1-mini")
print(f"OpenAI configured; vision model: {OPENAI_VISION_MODEL}")

OpenAI configured; vision model: gpt-4.1-mini


## Providing Images at the Start of the Agent's Execution

In this approach, images are passed to the agent at the start and stored as `task_images` alongside the task prompt. The agent then processes these images throughout its execution.  

Consider the case where Alfred wants to verify the identities of the superheroes attending the party. He already has a dataset of images from previous parties with the names of the guests. Given a new visitor's image, the agent can compare it with the existing dataset and make a decision about letting them in.  

In this case, a guest is trying to enter, and Alfred suspects that this visitor might be The Joker impersonating Wonder Woman. Alfred needs to verify their identity to prevent anyone unwanted from entering.  

Let’s build the example. First, the images are loaded. In this case, we use images from Wikipedia to keep the example minimal, but imagine the possible use-cases!

In [3]:
from io import BytesIO

import requests
from PIL import Image, UnidentifiedImageError

image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg",
    "https://upload.wikimedia.org/wikipedia/en/9/98/Joker_%28DC_Comics_character%29.jpg",
]

headers = {"User-Agent": "agents-course-vision-notebook/1.0"}
images = []
for url in image_urls:
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    try:
        images.append(Image.open(BytesIO(response.content)).convert("RGB"))
    except UnidentifiedImageError as exc:
        raise RuntimeError(f"The URL did not return a valid image: {url}") from exc

print(f"Loaded {len(images)} images.")

Loaded 2 images.


Now that we have the images, the OpenAI vision model will describe the visible costume and determine which of the two fictional characters it resembles.

In [4]:
from smolagents import CodeAgent, OpenAIServerModel

model = OpenAIServerModel(
    model_id=OPENAI_VISION_MODEL,
    api_key=os.environ["OPENAI_API_KEY"],
    max_tokens=1024,
)

In [5]:
agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=4,
    verbosity_level=2,
    executor_kwargs={"timeout_seconds": 60},
)

response = agent.run(
    """
    Describe only the visible costume, hair, and makeup in these images. Then state whether
    the costume more closely resembles The Joker or Wonder Woman, and briefly explain why.
    Treat this as fictional-character classification, not identification of a real person.
    """,
    images=images,
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Describe only the visible costume, hair, and makeup in these images. Then state whether                         │
│     the costume more closely resembles The Joker or Wonder Woman, and briefly explain why.                      │
│     Treat this as fictional-character classification, not identification of a real person.                      │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will describe the visible costume, hair, and makeup from each image. Then, I will analyze the           
characteristics and determine whether the costume more closely resembles The Joker or Wonder Woman. The distinctive
Joker traits are usually white face makeup, red lips, green hair, and typically purple or dark suits, while Wonder 
Woman is characterized by an Amazon warrior costume with red, blue, and gold colors, and no such clown-like makeup 
or green hair.                                                                                                     
                                                                                                                   
<code>                                                                                                             
# For the first image:                                                                                             
description_1 = {                                                                                                  
    "costume": "Purple jacket with a yellow shirt underneath, adorned with a large purple necktie or cravat.",     
    "hair": "Hair appears slicked back, dark in color with hints of green on the sides.",                          
    "makeup": "Face heavily painted white with blue eyeshadow around the eyes and bright red lipstick exaggerated  
into a smile."                                                                                                     
}                                                                                                                  
                                                                                                                   
# For the second image:                                                                                            
description_2 = {                                                                                                  
    "costume": "Dark suit jacket over a white shirt, with a large white flower pinned to the lapel.",              
    "hair": "Bright green, slicked back.",                                                                         
    "makeup": "White face with pronounced red lips stretched in a large sinister smile, dark eye makeup [48;2;13;17

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # For the first image:                                                                                           
  description_1 = {                                                                                                
      "costume": "Purple jacket with a yellow shirt underneath, adorned with a large purple necktie or cravat.",   
      "hair": "Hair appears slicked back, dark in color with hints of green on the sides.",                        
      "makeup": "Face heavily painted white with blue eyeshadow around the eyes and bright red lipstick            
  exaggerated into a smile."                                                                                       
  }                                                                                                                
                                                                                                                   
  # For the second image:                                                                                          
  description_2 = {                                                                                                
      "costume": "Dark suit jacket over a white shirt, with a large white flower pinned to the lapel.",            
      "hair": "Bright green, slicked back.",                                                                       
      "makeup": "White face with pronounced red lips stretched in a large sinister smile, dark eye makeup          
  accentuating the eyes."                                                                                          
  }                                                                                                                
                                                                                                                   
  # The descriptions here align closely with the known traits of the Joker character.                              
  character_classification = "The costume more closely resembles The Joker because both images feature the white   
  face makeup, exaggerated red lips, green hair, and predominantly purple or dark suits, which are signature       
  elements of The Joker character rather than Wonder Woman."                                                       
                                                                                                                   
  print(description_1)                                                                                             
  print(description_2)                                                                                             
  print(character_classification)                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'costume': 'Purple jacket with a yellow shirt underneath, adorned with a large purple necktie or cravat.', 'hair':
'Hair appears slicked back, dark in color with hints of green on the sides.', 'makeup': 'Face heavily painted white
with blue eyeshadow around the eyes and bright red lipstick exaggerated into a smile.'}
{'costume': 'Dark suit jacket over a white shirt, with a large white flower pinned to the lapel.', 'hair': 'Bright 
green, slicked back.', 'makeup': 'White face with pronounced red lips stretched in a large sinister smile, dark eye
makeup accentuating the eyes.'}
The costume more closely resembles The Joker because both images feature the white face makeup, exaggerated red 
lips, green hair, and predominantly purple or dark suits, which are signature elements of The Joker character 
rather than Wonder Woman.

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
The first image shows a costume with a purple jacket and a yellow shirt underneath, complemented by a large purple 
necktie or cravat. The hair is slicked back, dark with hints of green. The makeup consists of white face paint,    
blue eyeshadow around the eyes, and bright red exaggerated lips forming a wide smile.                              
                                                                                                                   
The second image shows a dark suit jacket over a white shirt, with a large white flower pinned to the lapel. The   
hair is bright green and slicked back. The makeup includes white face paint with pronounced red lips stretched into
a large sinister smile and dark eye makeup around the eyes.                                                        
                                                                                                                   
This costume more closely resembles The Joker. Both images show white painted faces, bold red lips in exaggerated  
smiles, green hair, and predominantly dark and purple suits—traits emblematic of The Joker rather than Wonder      
Woman’s warrior-style costume.                                                                                     

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
            Here is your code snippet:
            The first image shows a costume with a purple jacket and a yellow shirt underneath, complemented by a 
large purple necktie or cravat. The hair is slicked back, dark with hints of green. The makeup consists of white 
face paint, blue eyeshadow around the eyes, and bright red exaggerated lips forming a wide smile.

The second image shows a dark suit jacket over a white shirt, with a large white flower pinned to the lapel. The 
hair is bright green and slicked back. The makeup includes white face paint with pronounced red lips stretched into
a large sinister smile and dark eye makeup around the eyes.

This costume more closely resembles The Joker. Both images show white painted faces, bold red lips in exaggerated 
smiles, green hair, and predominantly dark and purple suits—traits emblematic of The Joker rather than Wonder 
Woman’s warrior-style costume.</code>
            Make sure to include code with the correct pattern, for instance:
            Thoughts: Your thoughts
            <code>
            # Your python code here
            </code>
Make sure to provide correct code blobs.

[Step 4: Duration 7.97 seconds| Input tokens: 21,903 | Output tokens: 941]

Reached max steps.

[Step 5: Duration 7.12 seconds| Input tokens: 29,350 | Output tokens: 1,264]

In [6]:
response

'Thoughts: The user wants a description of costume, hair, and makeup for two images along with a classification between The Joker or Wonder Woman. Since previous responses failed due to code formatting, this time I\'ll respond properly wrapped in a code block, as specified. I will describe each image and then provide a clear classification and explanation.\n\n<code>\n# Description and classification of the first image:\ndescription_image1 = {\n    "costume": "Purple jacket over a mustard yellow shirt with a large purple necktie or cravat.",\n    "hair": "Slicked back hair with dark color and subtle green highlights on the sides.",\n    "makeup": "White face paint, blue eyeshadow, and bright red lipstick exaggerated into a wide smile."\n}\n\n# Description and classification of the second image:\ndescription_image2 = {\n    "costume": "Dark suit jacket over a white shirt, adorned with a large white flower pinned to the lapel.",\n    "hair": "Bright green slicked-back hair.",\n    "makeup

In this case, the output reveals that the person is impersonating someone else, so we can prevent The Joker from entering the party!

## Providing Images with Dynamic Retrieval

This examples is provided as a `.py` file since it needs to be run locally since it'll browse the web. Go to the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course) for more details.